# DEMO: Introduction to the Medallion Architecture

In this demo you will see how data flows through the **Medallion Architecture** — the organising pattern for data in a Lakehouse:

```
Source Systems → Bronze (raw) → Silver (validated/cleaned) → Gold (business-ready) → Dashboards
```

You will:
1. Explore the **Bronze** layer — raw data exactly as it arrives, with quality issues
2. Explore the **Silver** layer — cleaned, deduplicated, and enriched
3. Explore the **Gold** layer — aggregated and ready for dashboards
4. Compare all three layers to see how data quality improves at each step

---

## Step 1: Run Setup

The cell below creates three schemas representing the Bronze, Silver, and Gold layers, and populates them with tables derived from sample data.

Click **Run** (▶) on the cell below.

In [0]:
%run ./Setup

---

## Step 2: The Bronze Layer (Raw Data)

Bronze is the **landing zone**. Data arrives here exactly as it comes from source systems — no transformations, no cleaning. It’s your safety net and audit trail.

In the setup, we ingested raw order data that simulates what real source systems deliver:
- **Duplicate rows** (from at-least-once delivery guarantees)
- **Numeric codes** instead of human-readable names
- **No standardised column names**

Let’s look at it.

In [0]:
# Switch to the Bronze schema
spark.sql(f"USE SCHEMA `{bronze_schema}`")
print(f"Now exploring: {catalog_name}.{bronze_schema}")

In [0]:
%sql
-- Preview the raw orders data - notice the source system column names and the _ingested_at timestamp
SELECT * FROM raw_orders LIMIT 10;

In [0]:
%sql
-- Check for duplicates: count total rows vs distinct order keys
SELECT 
  COUNT(*) AS total_rows,
  COUNT(DISTINCT o_orderkey) AS distinct_orders,
  COUNT(*) - COUNT(DISTINCT o_orderkey) AS duplicate_rows
FROM raw_orders;

Notice the duplicate count — this is typical of raw data. Source systems often deliver the same record more than once (at-least-once delivery). In the Bronze layer, we keep everything as-is.

Now look at the raw customer data:

In [0]:
%sql
-- Preview raw customers - notice c_nationkey is a numeric code, not a readable name
SELECT c_custkey, c_name, c_nationkey, c_mktsegment, c_acctbal 
FROM raw_customers 
LIMIT 10;

Notice `c_nationkey` is just a number (e.g., 15, 22). A business user looking at this data would have no idea what nation "15" means. The column names are also cryptic source-system codes (`c_custkey`, `c_acctbal`). This data is not useful for reporting — it needs cleaning.

---

## Step 3: The Silver Layer (Cleaned and Validated)

Silver is where data becomes **trustworthy**. The transformations applied here are:
- **Deduplication** — removing duplicate rows from source delivery issues
- **Standardised column names** — making them readable and consistent
- **Enrichment** — resolving numeric codes to human-readable values by joining reference tables

This is equivalent to the work Power Query does in Power BI — but done **once** by the data team, not repeated inside every report.

In [0]:
# Switch to the Silver schema
spark.sql(f"USE SCHEMA `{silver_schema}`")
print(f"Now exploring: {catalog_name}.{silver_schema}")

In [0]:
%sql
-- Preview Silver orders - notice clean column names and no duplicates
SELECT * FROM orders LIMIT 10;

**How this table was built:**

The setup created `orders` from Bronze `raw_orders` using this transformation:

```sql
CREATE TABLE silver.orders AS
SELECT DISTINCT                 -- removes all duplicate rows in one step
  o_orderkey      AS order_id,  -- cryptic source code → readable name
  o_custkey       AS customer_id,
  o_orderstatus   AS order_status,
  o_totalprice    AS total_price,
  o_orderdate     AS order_date,
  o_orderpriority AS order_priority,
  o_shippriority  AS ship_priority
FROM bronze.raw_orders
```

Two things happened in a single query:
- **`SELECT DISTINCT`** — eliminated the ~5% of duplicate rows that crept in from at-least-once delivery in Bronze
- **Column aliases (`AS`)** — every column renamed from source-system codes (`o_orderkey`, `o_totalprice`) to human-readable names (`order_id`, `total_price`)

Run the deduplication check below to see the row count drop.

In [0]:
%sql
-- Verify deduplication: total rows should equal distinct order count
SELECT COUNT(*) AS total_rows FROM orders;

Compare this count to the Bronze `raw_orders` table. The duplicates have been removed.

Now look at the enriched customers:

In [0]:
%sql
-- Preview Silver customers - nation codes are now readable names, region has been added
SELECT customer_id, customer_name, nation, region, market_segment, account_balance 
FROM customers 
LIMIT 10;

**How this table was built:**

The setup created `customers` by joining Bronze `raw_customers` against two reference tables to decode the `c_nationkey` numeric code:

```sql
CREATE TABLE silver.customers AS
SELECT
  c_custkey    AS customer_id,
  c_name       AS customer_name,
  n.n_name     AS nation,         -- c_nationkey 15 → "MOROCCO"
  r.r_name     AS region,         -- new column: did not exist in Bronze
  c_acctbal    AS account_balance,
  c_mktsegment AS market_segment
FROM bronze.raw_customers c
JOIN tpch.nation n ON c.c_nationkey = n.n_nationkey  -- resolves the numeric code to a name
JOIN tpch.region r ON n.n_regionkey = r.r_regionkey  -- chains from nation to add region
```

Three distinct transformations in one query:
- **Column renaming** — all `c_*` source codes replaced with readable names
- **Code resolution** — `c_nationkey` (a bare integer) is replaced with the actual nation name by joining the `nation` reference table
- **Enrichment** — a second join to the `region` table adds a `region` column that did not exist in Bronze at all

A data analyst could now work with this data comfortably. But for **dashboards**, we want something even more targeted.

---

## Step 4: The Gold Layer (Business-Ready)

Gold is the layer **your reports will love**. It is:
- **Aggregated** — pre-computed summaries optimised for specific business questions
- **Denormalised** — pre-joined so dashboards don’t need complex joins at query time
- **Documented** — table and column comments explain what everything means
- **Ready to query directly** — no need to import into a BI tool

In [0]:
# Switch to the Gold schema
spark.sql(f"USE SCHEMA `{gold_schema}`")
print(f"Now exploring: {catalog_name}.{gold_schema}")

In [0]:
%sql
-- Daily revenue summary - ready for a revenue dashboard
SELECT * FROM daily_revenue 
ORDER BY order_date DESC, total_revenue DESC 
LIMIT 15;

**How this table was built:**

The setup joined both Silver tables and collapsed them into a daily summary:

```sql
CREATE TABLE gold.daily_revenue AS
SELECT
  o.order_date,
  c.nation, c.region, c.market_segment,
  COUNT(o.order_id)            AS order_count,      -- pre-aggregated
  ROUND(SUM(o.total_price), 2) AS total_revenue,    -- pre-aggregated
  ROUND(AVG(o.total_price), 2) AS avg_order_value   -- pre-aggregated
FROM silver.orders o
JOIN silver.customers c ON o.customer_id = c.customer_id  -- pre-joined
GROUP BY o.order_date, c.nation, c.region, c.market_segment
```

`GROUP BY` collapsed many individual order rows into **one summary row per date/nation/segment combination** — which is why the Gold row count is far lower than Silver. The join between `orders` and `customers` is also done once here at build time, so dashboards never need to perform it at query time.

This table is **dashboard-ready**. A business analyst can immediately answer questions like:
- "What was our revenue yesterday by region?"
- "Which market segment is growing fastest?"

Now look at the customer-level metrics:

In [0]:
%sql
-- Customer lifetime spending - ready for customer analytics
SELECT 
  customer_name, 
  nation, 
  market_segment,
  lifetime_orders, 
  lifetime_revenue, 
  avg_order_value
FROM customer_spending 
ORDER BY lifetime_revenue DESC 
LIMIT 10;

**How this table was built:**

The setup rolled up every order per customer into a single lifetime summary row:

```sql
CREATE TABLE gold.customer_spending AS
SELECT
  c.customer_id, c.customer_name, c.nation, c.region, c.market_segment,
  COUNT(o.order_id)            AS lifetime_orders,    -- total orders ever placed
  ROUND(SUM(o.total_price), 2) AS lifetime_revenue,   -- total spend in USD
  ROUND(AVG(o.total_price), 2) AS avg_order_value,
  MIN(o.order_date)            AS first_order_date,   -- customer history span
  MAX(o.order_date)            AS last_order_date
FROM silver.customers c
JOIN silver.orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name, c.nation, c.region, c.market_segment
```

One row per customer — their entire order history compressed into metrics. `GROUP BY customer_id` is the key: instead of returning one row per order, it returns one row per customer with all their aggregates pre-computed. A dashboard reading this table never needs to touch the raw order detail.

---

## Step 5: Compare All Three Layers

Let’s put the three layers side by side to see how data quality and usability improve at each step.

In [0]:
# Compare row counts and purpose across all three layers
from pyspark.sql.functions import lit

bronze_count = spark.sql(f"SELECT COUNT(*) FROM `{catalog_name}`.`{bronze_schema}`.raw_orders").first()[0]
silver_count = spark.sql(f"SELECT COUNT(*) FROM `{catalog_name}`.`{silver_schema}`.orders").first()[0]
gold_count = spark.sql(f"SELECT COUNT(*) FROM `{catalog_name}`.`{gold_schema}`.daily_revenue").first()[0]

comparison = spark.createDataFrame([
    ("Bronze - raw_orders", bronze_count, "Raw ingestion", "Duplicates, cryptic column names, numeric codes"),
    ("Silver - orders", silver_count, "Cleaned & validated", "Deduplicated, standardised column names"),
    ("Gold - daily_revenue", gold_count, "Aggregated for dashboards", "Pre-joined, pre-aggregated, documented"),
], ["Layer & Table", "Row Count", "Purpose", "Characteristics"])

display(comparison)

Notice the pattern:
- **Bronze** has the most rows (including duplicates) — it’s the raw safety net
- **Silver** has fewer rows after deduplication — it’s the trusted, validated data
- **Gold** has the fewest rows because it’s aggregated — it’s optimised for fast dashboard queries

Each layer serves a different audience:

| Layer | Who Uses It | What For |
|---|---|---|
| **Bronze** | Data engineers, compliance teams | Debugging, auditing, replaying ingestion |
| **Silver** | Data analysts, data scientists | In-depth analysis, model training |
| **Gold** | Business analysts, executives, dashboards | Reporting, KPIs, self-service analytics |

---

## Summary

In this demo you have:

1. **Explored the Bronze layer** — raw data with duplicates and cryptic codes, exactly as source systems deliver it
2. **Explored the Silver layer** — cleaned, deduplicated, and enriched with readable names
3. **Explored the Gold layer** — aggregated and optimised, ready for dashboards to query directly
4. **Compared all three layers** — seeing how data quality improves progressively through the architecture


In [0]:
%sql
# Cleanup: uncomment and run the lines below ONLY if you want to remove the demo objects
# for schema in [bronze_schema, silver_schema, gold_schema]:
#     spark.sql(f"DROP SCHEMA IF EXISTS `{catalog_name}`.`{schema}` CASCADE")
# print("\u2713 Demo schemas removed")